In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("../Airlinedataset.csv")

target = "ticket_price"

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[target]),
    df[target],
    test_size=0.20,
    random_state=42
)

In [3]:
all_features = [
    "days_to_departure",
    "seats_remaining",
    "historical_demand",
    "competitor_price",
    "booking_velocity",
    "is_weekend",
    "flight_capacity"
]

no_seat_features = [
    "days_to_departure",
    "historical_demand",
    "competitor_price",
    "booking_velocity",
    "is_weekend"
]

market_features = [
    "days_to_departure",
    "historical_demand",
    "competitor_price",
    "booking_velocity",
    "is_weekend",
    "flight_capacity"
]

seat_features = [
    "seats_remaining",
    "flight_capacity"
]

In [4]:
def evaluate_feature_set(name, features):
    model = LinearRegression()

    model.fit(
        X_train[features],
        y_train
    )

    predictions = model.predict(
        X_test[features]
    )

    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    r2 = r2_score(y_test, predictions)

    return {
        "Feature Set": name,
        "Features": len(features),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [5]:
results = []

results.append(
    evaluate_feature_set(
        "All selected features",
        all_features
    )
)

results.append(
    evaluate_feature_set(
        "Without seat variables",
        no_seat_features
    )
)

results.append(
    evaluate_feature_set(
        "Market + capacity",
        market_features
    )
)

results.append(
    evaluate_feature_set(
        "Seat + capacity only",
        seat_features
    )
)

feature_results = pd.DataFrame(results)

feature_results

,Feature Set,Features,MAE,RMSE,R2
0,All selected features,7,228.291103,278.180900,0.958463
1,Without seat variables,5,812.236888,969.433629,0.495546
2,Market + capacity,6,795.754078,946.612244,0.519017
3,Seat + capacity only,2,773.969766,965.890170,0.499227


In [6]:
feature_results.sort_values("RMSE")

,Feature Set,Features,MAE,RMSE,R2
0,All selected features,7,228.291103,278.180900,0.958463
2,Market + capacity,6,795.754078,946.612244,0.519017
3,Seat + capacity only,2,773.969766,965.890170,0.499227
1,Without seat variables,5,812.236888,969.433629,0.495546


## Feature Ablation Findings

Feature ablation was performed to understand how dependent the baseline model is on different groups of variables.

The full seven-feature model achieved an R² of 0.9585, while removing groups of variables reduced performance substantially.

| Feature Set | MAE | RMSE | R² |
|---|---:|---:|---:|
| All selected features | 228.29 | 278.18 | 0.9585 |
| Market + capacity | 795.75 | 946.61 | 0.5190 |
| Seat + capacity | 773.97 | 965.89 | 0.4992 |
| Without seat variables | 812.24 | 969.43 | 0.4955 |

The results indicate that the model's predictive performance depends on the combined information provided by the selected feature set rather than one isolated feature group.

This also highlights an important limitation: the dataset does not contain timestamps or sequential booking histories. Therefore, the analysis cannot establish whether every feature would be available at a specific real-world pricing decision point.

The final system should therefore be presented as a data-driven pricing model and simulation based on the available flight-state variables, rather than as a complete production airline revenue-management system.